# Surrogate Factory — UCAirfoils
## Chapter 2. Data Acquisition / Generation
Objectives:
- Build a Sobol **Design of Experiments** over the airfoil design space
  (`metadata/SF_2_Data_Acquisition_Generation.yaml`).
- Evaluate every point with **NeuralFoil** to obtain CL, CD, CM and the upper
  and lower transition locations.

The design space has 22 entries: 20 sampled (alpha, reynolds, 8 upper and 8
lower Kulfan CST weights, TE thickness, LE weight), 2 derived (`alpha_rad`,
`log10_Re`) and 1 fixed (`mach = 0.3`).


### 0. Workflow initialisation

In [ ]:
from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow("pipeline_config.yaml")
workflow.resume()


### 2. Data Acquisition / Generation

In [ ]:
workflow.import_metadata(stage_name="SF_2_Data_Acquisition_Generation")

#### 2.1 Design of Experiments
Sobol sample of the bounded variables, then the derived and fixed columns, then the design-space constraints on `alpha_rad`.

In [ ]:
from data_acquisition.doe import define_doe
DoE = define_doe(workflow)
DoE.describe()


#### 2.2 Evaluate with NeuralFoil
`analysis_confidence` is NeuralFoil's own trust score for each point — SF_3 filters on it.

In [ ]:
from data_acquisition.doe import launch_sim
Dataset = launch_sim(workflow, DoE)
Dataset.head()


#### 2.3 Confidence distribution
Random Kulfan weight combinations produce many implausible sections; those come back with low confidence.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(Dataset["analysis_confidence"], bins=50, color="steelblue")
axes[0].axvline(0.5, color="red", ls="--", label="SF_3 threshold")
axes[0].set_xlabel("analysis_confidence"); axes[0].set_ylabel("count"); axes[0].legend()

for thr in (0.0, 0.5):
    sub = Dataset[Dataset["analysis_confidence"] >= thr]
    axes[1].hist(sub["CD"], bins=60, alpha=0.55, label=f"conf >= {thr}  (n={len(sub):,})")
axes[1].set_xlabel("CD"); axes[1].legend()
plt.tight_layout(); plt.show()

low = (Dataset["analysis_confidence"] < 0.5)
print(f"below 0.5: {low.sum():,} of {len(Dataset):,} ({low.mean():.1%})")
print(f"CD std — low confidence {Dataset.loc[low, 'CD'].std():.4f}   "
      f"high {Dataset.loc[~low, 'CD'].std():.4f}")


### Save

In [ ]:
workflow.save_data(Dataset, workflow.config["job_name"] + "_Raw.csv")
workflow.save_metadata()
